In [1]:
## -*- coding: utf-8 -*-
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from pandas import DataFrame, Series
from math import *
import seaborn as sns
import matplotlib.cm as cm
import collections
import glob
import sys
import time
from Bio import SeqIO
import random
import gzip
import time
import pickle
import subprocess
import re
mylen = np.vectorize(len)

%matplotlib inline

In [ ]:
### Define functions


'''
Functions for paired-end error calculation

- paired_end_error: calculates the error rate between a sequence and its complementary sequence.
    - seq: the original sequence.
    - comp_seq: the complementary sequence to compare against.
'''

def paired_end_error(seq, comp_seq):
    if len(seq) != len(comp_seq):
        return 1.0
    error=0
    for j, comp_base in enumerate(comp_seq[::-1]):
        if (comp_base=='A') & (seq[j]=='T'):
            continue
        elif (comp_base=='T') & (seq[j]=='A'):
            continue
        elif (comp_base=='G') & (seq[j]=='C'):
            continue
        elif (comp_base=='C') & (seq[j]=='G'):
            continue
        else:
            error+=1
    return error/len(seq)


'''
Functions for BLAST

- make_fasta: creates a FASTA file from a list of sequences.
    - seqs: list of sequences to include in the FASTA file.
    - Name: name of the FASTA file to create (without extension).
    - key: type of sequences ('RNA' or 'DNA') to determine whether to replace 'T' with 'U'.

- read_fasta: reads sequences from a FASTA file and returns them as a list.
    - path: path to the FASTA file to read.

- run_blast: creates a BLAST database from the provided sequences and runs BLAST to compare query sequences against the database.
    - query_seqs: list of sequences to use as queries in the BLAST search.
    - db_seqs: list of sequences to include in the BLAST database.
    - output_name: name of the output file to save BLAST results (without extension).
    - max_target_seqs: maximum number of target sequences to report for each query (default is 5).
    - num_threads: number of threads to use for BLAST (default is 1).
    - use_precomputed_db: whether to use a precomputed BLAST database (default is False).
    - db_name: name of the BLAST database to create or use (default is 'blast_db').
    - rm_db: whether to remove the BLAST database after running BLAST (default is False).

- read_blast_results: reads BLAST results from a TSV file and returns them as a DataFrame.
    - file_name: name of the BLAST results file to read (without extension).

- eval_blast_result: calculates query coverage and identity from BLAST results.
    - df_query: DataFrame containing the original query sequences and their lengths.
    - df_blast: DataFrame containing the BLAST results for the query sequences.
'''

def make_fasta(seqs, Name, key):
    if key=='RNA':
        seqs=[seq.replace('T', 'U') for seq in seqs]
    elif key=='DNA':
        seqs=seqs
    else:
        raise KeyError('Key must be "RNA" or "DNA".')
    seq_name=Name.rsplit('/')[-1]
    seq_list = []
    for i, seq in enumerate(np.array(seqs)):
        seq_list.append('>'+seq_name+'_'+str(i+1)+'\n')
        seq_list.append(seq + '\n')
    path = Name+'.fasta'
    open(path, "w").writelines(seq_list)

def read_fasta(path):
    seqs = []
    for record in SeqIO.parse(path, "fasta"):
        seqs.append(str(record.seq))
    return seqs

def run_blast(query_seqs, db_seqs, output_name, max_target_seqs=5, num_threads=1, use_precomputed_db=False, db_name='blast_db', rm_db=False):
    if not use_precomputed_db:
        make_fasta(db_seqs, db_name, key='DNA')
        subprocess.run(['mkdir', 'blast_db'], stdout=subprocess.DEVNULL)
        subprocess.run(['makeblastdb', '-in', db_name+'.fasta', '-dbtype', 'nucl', '-out', 'blast_db/'+db_name], check=True, stdout=subprocess.DEVNULL)
    make_fasta(query_seqs, 'query', key='DNA')
    subprocess.run(['blastn', '-query', 'query.fasta', '-task', 'blastn', '-db', 'blast_db/'+db_name, '-outfmt', '6', '-strand', 'plus',
                    '-max_target_seqs', str(max_target_seqs), '-out', output_name+'.tsv', '-num_threads', str(num_threads)], check=True, stdout=subprocess.DEVNULL)
    
    subprocess.run(['rm', 'query.fasta'], check=True)
    if not use_precomputed_db:
        subprocess.run(['rm', db_name+'.fasta'], check=True)
    if rm_db:
        subprocess.run(['rm', '-r', 'blast_db'], check=True)

def read_blast_results(file_name):
    columns=['query', 'subject_id', 'ident', 'align_len', 'mismatche', 'gap',
            'q_start', 'q_end', 's_start', 's_end', 'evalue', 'bit_score']
    df_blast = pd.read_csv(file_name+'.tsv', sep='\t', names=columns)
    return df_blast

def eval_blast_result(df_query, df_blast):
    seqs = df_query['seqs'].values
    lens = df_query['lens'].values
    qcover, ident = np.zeros(len(seqs)), np.zeros(len(seqs))
    query_ids = np.array([int(query.split('_')[-1]) for query in df_blast['query'].values])

    c = collections.Counter(query_ids)
    inds = []
    n = 0
    for v in c.values():
        inds += [n]
        n += v
    df_blast = df_blast.iloc[inds].reset_index(drop=True)
    new_inds = np.array(list(c.keys())) - 1
    new_inds = new_inds.astype(int)

    qcover[new_inds] = (df_blast['q_end'].values - df_blast['q_start'].values + 1) / lens[new_inds]
    qcover = qcover * 100
    ident[new_inds] = df_blast['ident'].values
    
    df = pd.concat([df_query, DataFrame({'qcover': qcover, 'ident': ident})], axis=1)
    return df

In [ ]:
### Before processing, run run_cutadapt.sh to trim adapters from the raw fq.gz files, and create fastq files for each series in data/trimmed/{name}/

In [ ]:
### Calculate error rate of paired-end sequencing and extract sequences with zero error, then count unique sequences and filter by length

name = 'N14'
Xs = [1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]
min_len, max_len = 15, 45

if 'data/raw_seqs_counts' not in glob.glob('data/*'):
    subprocess.run(['mkdir', 'data/raw_seqs_counts'], check=True)
if f'data/raw_seqs_counts/{name}' not in glob.glob('data/raw_seqs_counts/*'):
    subprocess.run(['mkdir', f'data/raw_seqs_counts/{name}'], check=True)

for k in Xs:
    start=time.time()
    seqs1, seqs2 = [], []
    with open(f'data/trimmed/{name}/trimmed_{k}_1.fq') as handle:
        for record in SeqIO.parse(handle, 'fastq'):
            seqs1.append(str(record.seq))
    seqs1=np.array(seqs1)
    with open(f'data/trimmed/{name}/trimmed_{k}_2.fq') as handle:
        for record in SeqIO.parse(handle, 'fastq'):
            seqs2.append(str(record.seq))
    seqs2=np.array(seqs2)
    
    errors=np.zeros(len(seqs1))
    for i in range(len(seqs1)):
        seq1=seqs1[i]
        seq2=seqs2[i]
        errors[i]=paired_end_error(seq1, seq2)
    inds=np.where(errors==0)[0]

    new_seqs=seqs1[inds]
    c = collections.Counter(new_seqs)
    seqs, counts = zip(*c.most_common())
    lens = mylen(seqs)
    valid_inds = np.where((lens>=min_len) & (lens<=max_len))[0]
    seqs = np.array(seqs)[valid_inds]
    counts = np.array(counts)[valid_inds]
    print(f'Retained {len(seqs)} sequences ({len(seqs)/len(seqs1)*100:.2f}%)')

    np.save(f'data/raw_seqs_counts/{name}/{k}_seqs.npy', seqs)
    np.save(f'data/raw_seqs_counts/{name}/{k}_counts.npy', counts)

    print(f'Elapsed time for {name}, k={k}: {time.time()-start:.2f}s')

In [ ]:
### Filter contaminants by BLASTing against marker sequences, RT primer and negative control sequences

name = 'N14'
Xs = [1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]

# Marker sequences and RT primer sequence
p100 ='GGGCGAATTGGAGCTCCACCGCGGTGGCGGCCGCTCTAGAACTAGTGGATCCTCAACTGTGAGGAGGCTCACGGACGCGAAGAACAGGCACGAATTCGAT' 
p50 = 'CGGTCATGGTGTATTTGCCCTTAAGCACGGCAGAAACTGCCTGCATCTCT'
p30 = 'AGAAGTCAGCAACTTGATTCCAGCAGTAGT'
pRT = 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCTTTTTTTTTTTT'
sp_seqs = [p100, p50, p30, pRT]

if name in ['N14', 'N14p', 'N20', 'N20p']:
    db_seqs = sp_seqs + list(np.load('data/raw_seqs_counts/NC/NC_1_seqs.npy', allow_pickle=True))
else:
    db_seqs = sp_seqs + list(np.load('data/raw_seqs_counts/NC/NC_0_seqs.npy', allow_pickle=True))

if 'data/filtered' not in glob.glob('data/*'):
    subprocess.run(['mkdir', 'data/filtered'], check=True)

ds = []
for k in Xs:
    start = time.time()
    seqs = np.load(f'data/raw_seqs_counts/{name}/{k}_seqs.npy', allow_pickle=True)
    counts = np.load(f'data/raw_seqs_counts/{name}/{k}_counts.npy', allow_pickle=True)
    lens = mylen(seqs)
    df = DataFrame({'seqs': seqs, 'counts': counts, 'lens': lens})

    if k == Xs[0]:
        use_precomputed_db, rm_db = False, False
    elif k == Xs[-1]:
        use_precomputed_db, rm_db = True, True
    else:
        use_precomputed_db, rm_db = True, False
    
    run_blast(seqs, db_seqs, output_name='blast_result', max_target_seqs=5, num_threads=8,
                use_precomputed_db=use_precomputed_db, db_name='blast_db', rm_db=rm_db)
    df_blast = read_blast_results('blast_result')
    df = eval_blast_result(df, df_blast)

    total_counts = df['counts'].sum()
    df = df[(df['qcover']<90) | (df['ident']<90)].reset_index(drop=True)
    contami_counts = total_counts - df['counts'].sum()

    print(f'Filtering contaminants for {name}, k={k}: {contami_counts} sequences ({contami_counts/total_counts*100:.2f}%)')
    df = df.loc[:, 'seqs':'lens']
    ds.append(df)
    print(f'Elapsed time for {name}, k={k}: {time.time()-start:.2f}s')

pickle.dump(ds, open(f'data/filtered/{name}_filtered.pkl', 'wb'))

In [ ]:
### Extract top sequences and create FASTA file for BLASTing against genomic databases

name = 'N14'

with open(f'data/filtered/{name}_filtered.pkl', 'rb') as handle:
    ds = pickle.load(handle)

if 'data/fasta' not in glob.glob('data/*'):
    subprocess.run(['mkdir', 'data/fasta'], check=True)

top_seqs = []
top_counts = []
for i in range(len(ds)):
    d = ds[i]
    if len(d) > 200:
        d = d[:200]
    top_seqs += d['seqs'].tolist()
    top_counts += d['counts'].tolist()
    # print(f'200th count {d.iloc[-1]["counts"]}')

if len(top_seqs) == 0:
    raise ValueError('No sequences found with the given threshold.')
df = DataFrame({'seqs': top_seqs, 'counts': top_counts})
df = df.groupby('seqs', as_index=False).sum()
df = df.sort_values('counts', ascending=False).reset_index(drop=True)

top_seqs = df['seqs'].values
top_counts = df['counts'].values
print(f'Top sequences {name}: {len(top_seqs)}')
make_fasta(top_seqs, f'data/fasta/{name}_top_seqs', key='DNA')

In [ ]:
### Before running next steps, run run_blastn.sh to BLAST against genomic databases, and save results as TSV files in data/tsv/

In [ ]:
### Read BLAST results and create potential contaminants db

name = 'N14'
threshold = 90

seqs = read_fasta(f'data/fasta/{name}_top_seqs.fasta')
df_query = DataFrame({'seqs': seqs, 'lens': mylen(seqs)})
df_blast = read_blast_results(f'data/tsv/{name}_top_seqs_result')
df = eval_blast_result(df_query, df_blast)
df_filtered = df[(df['qcover']>=threshold) & (df['ident']>=threshold)].reset_index(drop=True)
print(f'Contaminants in {name}: {len(df_filtered)}, {len(df_filtered)/len(df)*100:.2f}%')
make_fasta(df_filtered['seqs'].values, f'data/fasta/{name}_top_seqs_contaminants', key='DNA')

In [ ]:
### Filter contaminants by BLASTing against potential contaminants db

name = 'N14'

with open(f'data/filtered/{name}_filtered.pkl', 'rb') as handle:
    ds = pickle.load(handle)

ds_new = []
db_seqs = read_fasta(f'data/fasta/{name}_top_seqs_contaminants.fasta')
for i in range(len(ds)):
    start = time.time()
    df = ds[i]
    seqs = df['seqs'].values

    if i == 0:
        use_precomputed_db, rm_db = False, False
    elif i == (len(ds) - 1):
        use_precomputed_db, rm_db = True, True
    else:
        use_precomputed_db, rm_db = True, False
    
    run_blast(seqs, db_seqs, output_name='blast_result', max_target_seqs=5, num_threads=4,
                use_precomputed_db=use_precomputed_db, db_name='blast_db', rm_db=rm_db)
    df_blast = read_blast_results('blast_result')
    df = eval_blast_result(df, df_blast)

    total_counts = df['counts'].sum()
    df = df[(df['qcover']<90) | (df['ident']<90)].reset_index(drop=True)
    contami_counts = total_counts - df['counts'].sum()
    
    print(f'Filtering contaminants for {name}, i={i}: {contami_counts} sequences ({contami_counts/total_counts*100:.2f}%)')
    df = df.loc[:, 'seqs':'lens']
    ds_new.append(df)
    print(f'Elapsed time for {name}, i={i}: {time.time()-start:.2f}s')

pickle.dump(ds_new, open(f'data/filtered/{name}_filtered_blast.pkl', 'wb'))

#### Below is for only 2mer datasets (w/ AG(NNAG)5 or CU(NNCU) and Negative control)
Filtering sequences containing AG(NNAG)2 from w/ AG(NNAG)5 or CU(NNCU)2 from w/ CU(NNCU)5 to eliminate the influence of artifacts.

In [ ]:
with open(f'data/filtered/2mer_filtered_blast.pkl', 'rb') as handle:
    ds = pickle.load(handle)

ds_new = []
Rs = []
for i in range(len(ds)):
    d = ds[i]
    inds = []
    Rs.append(i)
    if i in [0, 1]:
        pattern = r'(AG.{2}){2}AG'
    elif i in [2, 3]:
        pattern = r'(CT.{2}){2}CT'
    else:
        continue
    for j, seq in enumerate(d['seqs'].values):
        if re.search(pattern, seq):
            inds.append(j)
    ds_new.append(d.drop(index=inds).reset_index(drop=True))

i = 4

d = ds[i]
inds = []
Rs.append(i)
pattern = r'(AG.{2}){2}AG'
for j, seq in enumerate(d['seqs'].values):
    if re.search(pattern, seq):
        inds.append(j)
ds_new.append(d.drop(index=inds).reset_index(drop=True))

d = ds[i]
inds = []
Rs.append(i)
pattern = r'(CT.{2}){2}CT'
for j, seq in enumerate(d['seqs'].values):
    if re.search(pattern, seq):
        inds.append(j)
ds_new.append(d.drop(index=inds).reset_index(drop=True))

pickle.dump(ds_new, open(f'data/filtered/2mer_filtered_blast_no_repeats_norm.pkl', 'wb'))